# Phase 2: Advanced Preprocessing & Data Engineering

**Objective:** Transform raw COCO annotations into a clean, YOLO-ready dataset optimized for YOLOv10 table detection.

---

## Pipeline Overview

| Step | Operation | Key Enhancement |
|------|-----------|----------------|
| 1 | Configuration & Setup | Kaggle paths, split ratios, seed |
| 2 | Load & Parse COCO JSON | Build DataFrames, category mapping |
| 3 | Semantic Label Filtering | Table-only (cat 1) + bbox expansion safety net |
| 4 | Coordinate Normalization | COCO → YOLO with clipping & sanitization |
| 5 | Document-Grouped Splitting | 85/10/5 split grouped by paper ID (no leakage) |
| 6 | YOLO Directory Export | Images + labels + data.yaml |
| 7 | Visual Verification | Overlay YOLO boxes on sample images |
| 8 | Summary Report | Statistics, integrity checks, completion checklist |

---

## Table of Contents

1. [Configuration & Setup](#config)
2. [Load & Parse COCO JSON](#load)
3. [Semantic Label Filtering](#filtering)
4. [Coordinate Normalization & Integrity](#normalization)
5. [Document-Grouped Dataset Splitting](#splitting)
6. [YOLO Directory Export](#export)
7. [Visual Verification](#verification)
8. [Summary Report & Completion Checklist](#summary)

---

<a id='config'></a>
## 1. Configuration & Setup

In [ ]:
import json
import os
import re
import shutil
import warnings
import textwrap
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =====================================================================
# PATHS  (Kaggle environment)
# =====================================================================
JSON_PATH   = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/annotations/Cells_Anotations_coco.json'
IMAGES_DIR  = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images'
OUTPUT_ROOT = '/kaggle/working/yolo_dataset'   # output directory for the YOLO dataset

# =====================================================================
# DATASET SPLIT RATIOS
# =====================================================================
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RANDOM_SEED = 42

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Split ratios must sum to 1.0"

# =====================================================================
# YOLO CLASS MAPPING
# =====================================================================
# COCO category_id 1 (table) --> YOLO class_id 0
COCO_TABLE_CAT_ID = 1
YOLO_TABLE_CLASS   = 0
CLASS_NAMES        = ['table']

# =====================================================================
# EXPANSION SAFETY-NET PARAMS
# =====================================================================
# Category 5 = "table projected row header" - might sit outside parent table
EXPANDABLE_CAT_ID  = 5
CONTAINMENT_TOL_PX = 5   # tolerance in pixels for containment check

# =====================================================================
# DOCUMENT ID REGEX
# =====================================================================
DOC_ID_PATTERN = re.compile(r'^(PMC\d+)_\d+\.jpg$')

# =====================================================================
# VISUALIZATION
# =====================================================================
TABLE_COLOR = '#E74C3C'
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Phase 2 configuration loaded.")
print(f"   Train / Val / Test = {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")
print(f"   Output root: {OUTPUT_ROOT}")
print(f"   Random seed: {RANDOM_SEED}")

<a id='load'></a>
## 2. Load & Parse COCO JSON

In [ ]:
# ============================================================
# Load COCO JSON
# ============================================================
with open(JSON_PATH, 'r') as f:
    coco_data = json.load(f)

images_list      = coco_data['images']
annotations_list = coco_data['annotations']
categories_list  = coco_data['categories']

print(f"   Loaded {len(images_list):,} images")
print(f"   Loaded {len(annotations_list):,} annotations")
print(f"   Loaded {len(categories_list)} categories")

# -------------------- Category Mapping --------------------
df_cats = pd.DataFrame(categories_list)
CAT_ID_TO_NAME = dict(zip(df_cats['id'], df_cats['name']))
CAT_NAME_TO_ID = dict(zip(df_cats['name'], df_cats['id']))

print("\n Categories:")
for _, row in df_cats.iterrows():
    marker = " <-- TARGET" if row['id'] == COCO_TABLE_CAT_ID else "     DROP"
    print(f"   {row['id']}: {row['name']} (parent: {row['supercategory']}) {marker}")

# -------------------- Images DataFrame --------------------
df_images = pd.DataFrame(images_list)
df_images = df_images.rename(columns={'id': 'image_id'})

# Extract document ID from filename  (PMC1064076_6.jpg --> PMC1064076)
df_images['doc_id'] = df_images['file_name'].apply(
    lambda f: DOC_ID_PATTERN.match(f).group(1) if DOC_ID_PATTERN.match(f) else f
)

# -------------------- Annotations DataFrame --------------------
df_anns = pd.DataFrame(annotations_list)
df_anns['bbox_x'] = df_anns['bbox'].apply(lambda b: b[0])
df_anns['bbox_y'] = df_anns['bbox'].apply(lambda b: b[1])
df_anns['bbox_w'] = df_anns['bbox'].apply(lambda b: b[2])
df_anns['bbox_h'] = df_anns['bbox'].apply(lambda b: b[3])

# Merge image metadata
df_anns = df_anns.merge(
    df_images[['image_id', 'file_name', 'width', 'height', 'doc_id']],
    on='image_id', how='left'
)
df_anns['category_name'] = df_anns['category_id'].map(CAT_ID_TO_NAME)

print(f"\n DataFrames built successfully.")
print(f"   df_images shape: {df_images.shape}")
print(f"   df_anns shape:   {df_anns.shape}")
print(f"   Unique documents: {df_images['doc_id'].nunique()}")

# Quick annotation breakdown
print("\n Annotations by category:")
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    count = (df_anns['category_id'] == cat_id).sum()
    print(f"   {cat_name}: {count:,}")

<a id='filtering'></a>
## 3. Semantic Label Filtering

**Goal:** Keep only `category_id == 1` (table) annotations. Before dropping children, check if any `table projected row header` (cat 5) boxes sit outside their parent table box. If so, expand the table box to encompass them — maximizing the containment area.

Phase 1 found 0 containment violations. This cell acts as a **safety net** for robustness.

In [ ]:
# ============================================================
# Step 3: Semantic Label Filtering + Table BBox Expansion
# ============================================================

print("="*60)
print(" SEMANTIC LABEL FILTERING")
print("="*60)

n_total_anns = len(df_anns)
n_table_anns_before = (df_anns['category_id'] == COCO_TABLE_CAT_ID).sum()
n_dropped = n_total_anns - n_table_anns_before

print(f"\n Before filtering:")
print(f"   Total annotations: {n_total_anns:,}")
print(f"   Table annotations (cat {COCO_TABLE_CAT_ID}): {n_table_anns_before:,}")
print(f"   Non-table annotations to drop: {n_dropped:,}")

# ============================================================
# SAFETY NET: Expand table boxes to encompass stray projected row headers
# ============================================================
print(f"\n Running table bbox expansion safety net...")
print(f"   Checking if any cat {EXPANDABLE_CAT_ID} (table projected row header) boxes")
print(f"   fall outside their parent table bboxes (tolerance: {CONTAINMENT_TOL_PX}px)...")

expansion_count = 0
expansion_log = []

# Work on a copy of table annotations to allow in-place bbox modification
df_tables_work = df_anns[df_anns['category_id'] == COCO_TABLE_CAT_ID].copy()
df_projected = df_anns[df_anns['category_id'] == EXPANDABLE_CAT_ID].copy()

# Build a lookup: image_id -> list of table annotation indices (in df_tables_work)
table_idx_by_image = defaultdict(list)
for idx, row in df_tables_work.iterrows():
    table_idx_by_image[row['image_id']].append(idx)

for _, proj_row in df_projected.iterrows():
    img_id = proj_row['image_id']
    px, py, pw, ph = proj_row['bbox_x'], proj_row['bbox_y'], proj_row['bbox_w'], proj_row['bbox_h']
    p_x2 = px + pw
    p_y2 = py + ph
    
    table_indices = table_idx_by_image.get(img_id, [])
    if not table_indices:
        continue  # No table on this image (shouldn't happen per Phase 1)
    
    # Check containment in each table box
    contained = False
    best_table_idx = None
    best_overlap = -1
    
    for t_idx in table_indices:
        t = df_tables_work.loc[t_idx]
        tx, ty, tw, th = t['bbox_x'], t['bbox_y'], t['bbox_w'], t['bbox_h']
        t_x2 = tx + tw
        t_y2 = ty + th
        
        # Containment check with tolerance
        if (px >= tx - CONTAINMENT_TOL_PX and
            py >= ty - CONTAINMENT_TOL_PX and
            p_x2 <= t_x2 + CONTAINMENT_TOL_PX and
            p_y2 <= t_y2 + CONTAINMENT_TOL_PX):
            contained = True
            break
        
        # Track the table with max overlap (for expansion target)
        inter_x1 = max(px, tx)
        inter_y1 = max(py, ty)
        inter_x2 = min(p_x2, t_x2)
        inter_y2 = min(p_y2, t_y2)
        overlap = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
        
        if overlap > best_overlap:
            best_overlap = overlap
            best_table_idx = t_idx
    
    if not contained and best_table_idx is not None:
        # Expand the nearest table box to encompass this projected row header
        t = df_tables_work.loc[best_table_idx]
        old_tx, old_ty = t['bbox_x'], t['bbox_y']
        old_tx2 = old_tx + t['bbox_w']
        old_ty2 = old_ty + t['bbox_h']
        
        new_tx  = min(old_tx,  px)
        new_ty  = min(old_ty,  py)
        new_tx2 = max(old_tx2, p_x2)
        new_ty2 = max(old_ty2, p_y2)
        
        # Clamp to image boundaries
        img_w = proj_row['width']
        img_h = proj_row['height']
        new_tx  = max(0, new_tx)
        new_ty  = max(0, new_ty)
        new_tx2 = min(img_w, new_tx2)
        new_ty2 = min(img_h, new_ty2)
        
        new_tw = new_tx2 - new_tx
        new_th = new_ty2 - new_ty
        
        # Update the table bbox in-place
        df_tables_work.at[best_table_idx, 'bbox_x'] = new_tx
        df_tables_work.at[best_table_idx, 'bbox_y'] = new_ty
        df_tables_work.at[best_table_idx, 'bbox_w'] = new_tw
        df_tables_work.at[best_table_idx, 'bbox_h'] = new_th
        df_tables_work.at[best_table_idx, 'bbox'] = [new_tx, new_ty, new_tw, new_th]
        
        expansion_count += 1
        expansion_log.append({
            'image_id': img_id,
            'file_name': proj_row['file_name'],
            'table_ann_id': int(t['id']),
            'proj_ann_id': int(proj_row['id']),
            'old_bbox': [old_tx, old_ty, t['bbox_w'], t['bbox_h']],
            'new_bbox': [new_tx, new_ty, new_tw, new_th]
        })

# Report expansion results
if expansion_count > 0:
    print(f"\n   Table boxes expanded: {expansion_count}")
    print(f"   Expansion log (first 5):")
    for entry in expansion_log[:5]:
        print(f"      {entry['file_name']}: table ann {entry['table_ann_id']}")
        print(f"         old: {[round(v,1) for v in entry['old_bbox']]}")
        print(f"         new: {[round(v,1) for v in entry['new_bbox']]}")
else:
    print(f"\n   No expansions needed. All projected row headers are within parent tables.")

# ============================================================
# Final filtered DataFrame: table-only
# ============================================================
df_tables = df_tables_work.copy()

print(f"\n After filtering:")
print(f"   Table annotations retained: {len(df_tables):,}")
print(f"   Annotations dropped: {n_dropped:,} ({n_dropped/n_total_anns*100:.1f}%)")
print(f"   Table boxes expanded: {expansion_count}")

# How many images have at least one table?
images_with_tables = set(df_tables['image_id'].unique())
images_without_tables = set(df_images['image_id'].unique()) - images_with_tables
print(f"\n   Images WITH table annotations: {len(images_with_tables):,}")
print(f"   Images WITHOUT table annotations: {len(images_without_tables):,}")
print(f"   (Images without tables will get empty label files)")

<a id='normalization'></a>
## 4. Coordinate Normalization & Integrity

Convert COCO `[x, y, w, h]` pixel coordinates to YOLO normalized format:

$$\text{x\_center} = \frac{x_{\min} + x_{\max}}{2 \cdot W}, \quad \text{y\_center} = \frac{y_{\min} + y_{\max}}{2 \cdot H}$$

$$\text{width} = \frac{x_{\max} - x_{\min}}{W}, \quad \text{height} = \frac{y_{\max} - y_{\min}}{H}$$

**Sanitization:**
- Clip all edge coordinates to `[0, 1]` **before** computing center/size (geometrically correct)
- Discard boxes with `width <= 0` or `height <= 0` after clipping

In [ ]:
# ============================================================
# Step 4: COCO -> YOLO Coordinate Conversion with Sanitization
# ============================================================

print("="*60)
print(" COORDINATE NORMALIZATION & INTEGRITY")
print("="*60)

n_before = len(df_tables)
n_clipped = 0

yolo_records = []  # Each entry: (image_id, file_name, yolo_line_str)

for _, row in df_tables.iterrows():
    img_w = row['width']
    img_h = row['height']
    bx, by, bw, bh = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
    
    # Compute raw normalized edge coordinates
    x_min_raw = bx / img_w
    y_min_raw = by / img_h
    x_max_raw = (bx + bw) / img_w
    y_max_raw = (by + bh) / img_h
    
    # Clip edges to [0, 1]
    x_min = np.clip(x_min_raw, 0.0, 1.0)
    y_min = np.clip(y_min_raw, 0.0, 1.0)
    x_max = np.clip(x_max_raw, 0.0, 1.0)
    y_max = np.clip(y_max_raw, 0.0, 1.0)
    
    # Track if clipping occurred
    if (x_min != x_min_raw or y_min != y_min_raw or
        x_max != x_max_raw or y_max != y_max_raw):
        n_clipped += 1
    
    # Derive YOLO center + size from clipped edges
    w_norm = x_max - x_min
    h_norm = y_max - y_min
    
    # Discard degenerate boxes
    if w_norm <= 0 or h_norm <= 0:
        continue
    
    x_center = (x_min + x_max) / 2.0
    y_center = (y_min + y_max) / 2.0
    
    yolo_line = f"{YOLO_TABLE_CLASS} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
    
    yolo_records.append({
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'ann_id': row['id'],
        'yolo_line': yolo_line,
        'x_center': x_center,
        'y_center': y_center,
        'w_norm': w_norm,
        'h_norm': h_norm
    })

df_yolo = pd.DataFrame(yolo_records)
n_after = len(df_yolo)
n_discarded = n_before - n_after

print(f"\n Conversion Results:")
print(f"   Input table annotations: {n_before:,}")
print(f"   Boxes clipped to [0,1]:  {n_clipped}")
print(f"   Boxes discarded (w<=0 or h<=0): {n_discarded}")
print(f"   Valid YOLO labels: {n_after:,}")

# -------------------- Sanity Checks --------------------
print(f"\n Coordinate Range Verification:")
for col in ['x_center', 'y_center', 'w_norm', 'h_norm']:
    vmin = df_yolo[col].min()
    vmax = df_yolo[col].max()
    status = "OK" if 0.0 <= vmin and vmax <= 1.0 else "FAIL"
    print(f"   {col:>10s}: [{vmin:.6f}, {vmax:.6f}]  {status}")

# -------------------- Distribution Visualization --------------------
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, col, label in zip(axes, 
                          ['x_center', 'y_center', 'w_norm', 'h_norm'],
                          ['X Center', 'Y Center', 'Width', 'Height']):
    ax.hist(df_yolo[col], bins=40, color='#3498DB', edgecolor='black', alpha=0.7)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'YOLO {label} Distribution', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1)

plt.suptitle('Normalized YOLO Coordinate Distributions', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# -------------------- Sample Output --------------------
print(f"\n Sample YOLO label lines (first 5):")
for _, r in df_yolo.head(5).iterrows():
    print(f"   {r['file_name']}: {r['yolo_line']}")

<a id='splitting'></a>
## 5. Document-Grouped Dataset Splitting

**Critical:** Split by **document ID** (e.g., `PMC1064076`), not by individual image. Pages from the same paper share identical table styles — placing them across splits causes data leakage and inflates validation scores.

Strategy: `GroupShuffleSplit` with document ID as the group key.
- **Pass 1:** 85% train vs 15% temp
- **Pass 2:** Split temp into ~66.7% val / ~33.3% test → ~10% val, ~5% test overall

In [ ]:
# ============================================================
# Step 5: Document-Grouped Dataset Splitting
# ============================================================

print("="*60)
print(" DOCUMENT-GROUPED DATASET SPLITTING")
print("="*60)

# Build image-level DataFrame with document groups
# (includes ALL images, even those without table annotations)
df_split = df_images[['image_id', 'file_name', 'doc_id']].copy()

# Count table annotations per image
labels_per_image = df_yolo.groupby('image_id').size().reset_index(name='n_labels')
df_split = df_split.merge(labels_per_image, on='image_id', how='left')
df_split['n_labels'] = df_split['n_labels'].fillna(0).astype(int)

n_docs = df_split['doc_id'].nunique()
print(f"\n   Total images: {len(df_split):,}")
print(f"   Unique documents: {n_docs}")
print(f"   Images with tables: {(df_split['n_labels'] > 0).sum():,}")
print(f"   Images without tables: {(df_split['n_labels'] == 0).sum():,}")

# ============================================================
# Pass 1: Train vs Temp (val + test)
# ============================================================
temp_ratio = VAL_RATIO + TEST_RATIO  # 0.15

gss1 = GroupShuffleSplit(n_splits=1, test_size=temp_ratio, random_state=RANDOM_SEED)
train_idx, temp_idx = next(gss1.split(df_split, groups=df_split['doc_id']))

df_train = df_split.iloc[train_idx].copy()
df_temp  = df_split.iloc[temp_idx].copy()

# ============================================================
# Pass 2: Val vs Test from temp
# ============================================================
test_from_temp = TEST_RATIO / temp_ratio  # 0.05/0.15 = 0.3333

gss2 = GroupShuffleSplit(n_splits=1, test_size=test_from_temp, random_state=RANDOM_SEED)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['doc_id']))

df_val  = df_temp.iloc[val_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

# Assign split labels
df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

df_all_splits = pd.concat([df_train, df_val, df_test], ignore_index=True)

# ============================================================
# Validation: Zero Document Overlap
# ============================================================
train_docs = set(df_train['doc_id'].unique())
val_docs   = set(df_val['doc_id'].unique())
test_docs  = set(df_test['doc_id'].unique())

leak_tv = train_docs & val_docs
leak_tt = train_docs & test_docs
leak_vt = val_docs & test_docs

print(f"\n" + "-"*60)
print(f" SPLIT STATISTICS")
print(f"-"*60)

for name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n_imgs = len(df_s)
    n_lbls = df_s['n_labels'].sum()
    n_doc  = df_s['doc_id'].nunique()
    pct    = n_imgs / len(df_split) * 100
    density = n_lbls / n_imgs if n_imgs > 0 else 0
    print(f"\n   {name}:")
    print(f"      Images:      {n_imgs:,} ({pct:.1f}%)")
    print(f"      Labels:      {n_lbls:,}")
    print(f"      Documents:   {n_doc}")
    print(f"      Labels/Img:  {density:.2f}")

print(f"\n" + "-"*60)
print(f" DATA LEAKAGE CHECK")
print(f"-"*60)
print(f"   Train-Val overlap:  {len(leak_tv)} docs  {'OK' if len(leak_tv)==0 else 'LEAK DETECTED!'}")
print(f"   Train-Test overlap: {len(leak_tt)} docs  {'OK' if len(leak_tt)==0 else 'LEAK DETECTED!'}")
print(f"   Val-Test overlap:   {len(leak_vt)} docs  {'OK' if len(leak_vt)==0 else 'LEAK DETECTED!'}")

assert len(leak_tv) == 0, "Data leakage between Train and Val!"
assert len(leak_tt) == 0, "Data leakage between Train and Test!"
assert len(leak_vt) == 0, "Data leakage between Val and Test!"

# -------------------- Visualization --------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Split distribution (pie)
ax1 = axes[0]
sizes = [len(df_train), len(df_val), len(df_test)]
labels = [f'Train\n{sizes[0]:,}', f'Val\n{sizes[1]:,}', f'Test\n{sizes[2]:,}']
colors = ['#3498DB', '#2ECC71', '#F39C12']
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='black', linewidth=0.5))
ax1.set_title('Image Split Distribution', fontsize=12, fontweight='bold')

# Labels per split (bar)
ax2 = axes[1]
split_names = ['Train', 'Val', 'Test']
split_labels = [df_train['n_labels'].sum(), df_val['n_labels'].sum(), df_test['n_labels'].sum()]
bars = ax2.bar(split_names, split_labels, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Number of Labels', fontsize=11)
ax2.set_title('Table Labels per Split', fontsize=12, fontweight='bold')
for bar, val in zip(bars, split_labels):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:,}', ha='center', va='bottom', fontsize=10)

# Docs per split (bar)
ax3 = axes[2]
split_docs = [len(train_docs), len(val_docs), len(test_docs)]
bars = ax3.bar(split_names, split_docs, color=colors, edgecolor='black', linewidth=0.5)
ax3.set_ylabel('Number of Documents', fontsize=11)
ax3.set_title('Unique Documents per Split', fontsize=12, fontweight='bold')
for bar, val in zip(bars, split_docs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

<a id='export'></a>
## 6. YOLO Directory Export

Creates the standard YOLO directory structure:
```
yolo_dataset/
├── data.yaml
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

In [ ]:
# ============================================================
# Step 6: YOLO Directory Creation & Export
# ============================================================

print("="*60)
print(" YOLO DIRECTORY EXPORT")
print("="*60)

start_time = time.time()

# -------------------- Create Directory Structure --------------------
# Clean previous output if exists
if os.path.exists(OUTPUT_ROOT):
    shutil.rmtree(OUTPUT_ROOT)
    print(f"\n   Removed existing output directory.")

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_ROOT, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT, 'labels', split), exist_ok=True)

print(f"   Created directory structure at: {OUTPUT_ROOT}")

# -------------------- Build YOLO labels lookup --------------------
# {file_name: [yolo_line1, yolo_line2, ...]}
labels_by_file = defaultdict(list)
for _, r in df_yolo.iterrows():
    labels_by_file[r['file_name']].append(r['yolo_line'])

# -------------------- Export Images & Labels --------------------
export_stats = {'train': {'images': 0, 'labels': 0, 'empty': 0},
                'val':   {'images': 0, 'labels': 0, 'empty': 0},
                'test':  {'images': 0, 'labels': 0, 'empty': 0}}

errors = []

for _, row in df_all_splits.iterrows():
    split     = row['split']
    file_name = row['file_name']
    
    # --- Copy image ---
    src_path = os.path.join(IMAGES_DIR, file_name)
    dst_path = os.path.join(OUTPUT_ROOT, 'images', split, file_name)
    
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        export_stats[split]['images'] += 1
    else:
        errors.append(f"Image not found: {src_path}")
        continue
    
    # --- Write label file ---
    label_name = os.path.splitext(file_name)[0] + '.txt'
    label_path = os.path.join(OUTPUT_ROOT, 'labels', split, label_name)
    
    lines = labels_by_file.get(file_name, [])
    
    with open(label_path, 'w') as f:
        if lines:
            f.write('\n'.join(lines) + '\n')
            export_stats[split]['labels'] += len(lines)
        # else: empty file (image with no tables)
    
    if not lines:
        export_stats[split]['empty'] += 1

# -------------------- Generate data.yaml --------------------
yaml_content = textwrap.dedent(f"""\
    # YOLO Dataset Configuration - Table Detection
    # Auto-generated by Phase 2 Pipeline
    
    path: {OUTPUT_ROOT}
    train: images/train
    val: images/val
    test: images/test
    
    # Classes
    nc: {len(CLASS_NAMES)}
    names: {CLASS_NAMES}
""")

yaml_path = os.path.join(OUTPUT_ROOT, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

elapsed = time.time() - start_time

# -------------------- Report --------------------
print(f"\n" + "-"*60)
print(f" EXPORT SUMMARY")
print(f"-"*60)

total_imgs = 0
total_lbls = 0
total_empty = 0

for split in ['train', 'val', 'test']:
    s = export_stats[split]
    print(f"\n   {split.upper()}:")
    print(f"      Images copied:     {s['images']:,}")
    print(f"      Label lines:       {s['labels']:,}")
    print(f"      Empty label files:  {s['empty']}")
    total_imgs += s['images']
    total_lbls += s['labels']
    total_empty += s['empty']

print(f"\n   TOTAL:")
print(f"      Images: {total_imgs:,}")
print(f"      Labels: {total_lbls:,}")
print(f"      Empty label files: {total_empty}")

if errors:
    print(f"\n   Errors ({len(errors)}):")
    for e in errors[:10]:
        print(f"      {e}")
else:
    print(f"\n   No errors during export.")

print(f"\n   data.yaml written to: {yaml_path}")
print(f"   Export completed in {elapsed:.1f}s")

# -------------------- File Count Integrity Check --------------------
print(f"\n" + "-"*60)
print(f" FILE COUNT INTEGRITY CHECK")
print(f"-"*60)

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(OUTPUT_ROOT, 'images', split)
    lbl_dir = os.path.join(OUTPUT_ROOT, 'labels', split)
    n_img_files = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
    n_lbl_files = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
    match = "OK" if n_img_files == n_lbl_files else "MISMATCH!"
    print(f"   {split:>5s}: {n_img_files} images, {n_lbl_files} labels  {match}")

# Print data.yaml content
print(f"\n" + "-"*60)
print(f" data.yaml")
print(f"-"*60)
print(yaml_content)

<a id='verification'></a>
## 7. Visual Verification

Load sample images from each split, parse their `.txt` label files, convert YOLO normalized coordinates back to pixel coordinates, and overlay bounding boxes. This is the **end-to-end correctness check**.

In [ ]:
# ============================================================
# Step 7: Visual Verification  (YOLO -> pixel overlay)
# ============================================================

print("="*60)
print(" VISUAL VERIFICATION")
print("="*60)

def yolo_to_pixel(x_center, y_center, w_norm, h_norm, img_w, img_h):
    """Convert YOLO normalized coords back to pixel [x, y, w, h]."""
    w_px = w_norm * img_w
    h_px = h_norm * img_h
    x_px = x_center * img_w - w_px / 2
    y_px = y_center * img_h - h_px / 2
    return x_px, y_px, w_px, h_px

def parse_yolo_label(label_path):
    """Parse a YOLO .txt label file and return list of (class_id, xc, yc, w, h)."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            boxes.append((cls, xc, yc, w, h))
    return boxes

# Select 2 random images from each split (that have labels)
np.random.seed(RANDOM_SEED)
sample_images = []

for split_name, df_s in [('train', df_train), ('val', df_val), ('test', df_test)]:
    # Prefer images with tables for more informative visualization
    with_labels = df_s[df_s['n_labels'] > 0]
    if len(with_labels) >= 2:
        chosen = with_labels.sample(2, random_state=RANDOM_SEED)
    else:
        chosen = df_s.sample(min(2, len(df_s)), random_state=RANDOM_SEED)
    
    for _, row in chosen.iterrows():
        sample_images.append((split_name, row['file_name']))

n_samples = len(sample_images)
n_cols = min(3, n_samples)
n_rows = (n_samples + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
if n_samples == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)

for idx, (split_name, file_name) in enumerate(sample_images):
    r, c = idx // n_cols, idx % n_cols
    ax = axes[r, c]
    
    img_path = os.path.join(OUTPUT_ROOT, 'images', split_name, file_name)
    label_name = os.path.splitext(file_name)[0] + '.txt'
    label_path = os.path.join(OUTPUT_ROOT, 'labels', split_name, label_name)
    
    if os.path.exists(img_path):
        img = Image.open(img_path)
        img_w, img_h = img.size
        ax.imshow(img, cmap='gray')
        
        # Parse and overlay YOLO boxes
        boxes = parse_yolo_label(label_path)
        for cls_id, xc, yc, w, h in boxes:
            px, py, pw, ph = yolo_to_pixel(xc, yc, w, h, img_w, img_h)
            rect = Rectangle((px, py), pw, ph, linewidth=2.5,
                             edgecolor=TABLE_COLOR, facecolor='none',
                             linestyle='-')
            ax.add_patch(rect)
            ax.text(px, py - 8, f'table ({w*img_w:.0f}x{h*img_h:.0f}px)',
                    color=TABLE_COLOR, fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        
        ax.set_title(f'[{split_name.upper()}] {file_name}\n{len(boxes)} table(s)', fontsize=10)
    else:
        ax.text(0.5, 0.5, f'Image not found:\n{file_name}',
                ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

# Hide unused subplots
for idx in range(n_samples, n_rows * n_cols):
    r, c = idx // n_cols, idx % n_cols
    axes[r, c].axis('off')

fig.suptitle('Visual Verification: YOLO Labels Overlaid on Images',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n   Visual verification complete. Inspect the overlaid boxes above.")
print("   Red rectangles should tightly enclose each table in the image.")

<a id='summary'></a>
## 8. Summary Report & Completion Checklist

In [ ]:
# ============================================================
# Step 8: Summary Report & Completion Checklist
# ============================================================

print("="*70)
print(" PHASE 2 SUMMARY REPORT")
print("="*70)

# -------------------- Pipeline Summary --------------------
print("\n" + "-"*70)
print(" PIPELINE STATISTICS")
print("-"*70)

summary_rows = [
    ['Total COCO annotations (input)',        f'{n_total_anns:,}'],
    ['Non-table annotations dropped',         f'{n_dropped:,}'],
    ['Table annotations (cat 1) retained',    f'{n_table_anns_before:,}'],
    ['Table bboxes expanded (safety net)',     f'{expansion_count}'],
    ['Boxes clipped to [0,1]',                f'{n_clipped}'],
    ['Boxes discarded (degenerate)',           f'{n_discarded}'],
    ['Valid YOLO labels (output)',             f'{n_after:,}'],
    ['', ''],
    ['Total images',                          f'{len(df_split):,}'],
    ['Unique documents',                      f'{n_docs}'],
    ['Train images',                          f'{len(df_train):,} ({len(df_train)/len(df_split)*100:.1f}%)'],
    ['Val images',                            f'{len(df_val):,} ({len(df_val)/len(df_split)*100:.1f}%)'],
    ['Test images',                           f'{len(df_test):,} ({len(df_test)/len(df_split)*100:.1f}%)'],
    ['Train documents',                       f'{len(train_docs)}'],
    ['Val documents',                         f'{len(val_docs)}'],
    ['Test documents',                        f'{len(test_docs)}'],
    ['Document overlap (leakage)',            f'None'],
]

df_summary = pd.DataFrame(summary_rows, columns=['Metric', 'Value'])
display(df_summary.style.hide(axis='index'))

# -------------------- Annotation Reduction Funnel --------------------
print("\n" + "-"*70)
print(" ANNOTATION REDUCTION FUNNEL")
print("-"*70)

funnel_stages = [
    ('Raw COCO annotations',    n_total_anns),
    ('After semantic filtering', n_table_anns_before),
    ('After bbox expansion',     n_table_anns_before),  # same count, expanded coords
    ('After sanitization',       n_after),
]

fig, ax = plt.subplots(figsize=(10, 4))
stage_names = [s[0] for s in funnel_stages]
stage_counts = [s[1] for s in funnel_stages]
colors_funnel = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']

bars = ax.barh(stage_names[::-1], stage_counts[::-1], color=colors_funnel[::-1],
               edgecolor='black', linewidth=0.5, height=0.6)
ax.set_xlabel('Number of Annotations', fontsize=12)
ax.set_title('Annotation Reduction Funnel', fontsize=13, fontweight='bold')

for bar, val in zip(bars, stage_counts[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, max(stage_counts) * 1.15)
plt.tight_layout()
plt.show()

# -------------------- YOLO Label Statistics --------------------
print("\n" + "-"*70)
print(" YOLO LABEL COORDINATE STATISTICS")
print("-"*70)

yolo_stats = df_yolo[['x_center', 'y_center', 'w_norm', 'h_norm']].describe()
display(yolo_stats.round(6))

# -------------------- Final Integrity Checks --------------------
print("\n" + "="*70)
print(" PHASE 2 COMPLETION CHECKLIST")
print("="*70)

# Compute checks
all_coords_valid = (df_yolo[['x_center','y_center','w_norm','h_norm']].min().min() >= 0 and
                    df_yolo[['x_center','y_center','w_norm','h_norm']].max().max() <= 1)

checklist = [
    ("Semantic filtering (table-only)",          True),
    ("Table bbox expansion safety net",          True),
    ("COCO -> YOLO coordinate conversion",       True),
    ("Coordinate clipping [0,1]",                all_coords_valid),
    ("Degenerate box removal",                   True),
    ("Document-grouped splitting (no leakage)",  len(leak_tv) == 0 and len(leak_tt) == 0 and len(leak_vt) == 0),
    ("Train/Val/Test directories created",       all(os.path.isdir(os.path.join(OUTPUT_ROOT, 'images', s)) for s in ['train','val','test'])),
    ("Label files exported",                     total_lbls > 0),
    ("Image-label count match",                  all(
        len(os.listdir(os.path.join(OUTPUT_ROOT, 'images', s))) == 
        len(os.listdir(os.path.join(OUTPUT_ROOT, 'labels', s)))
        for s in ['train','val','test']
    )),
    ("data.yaml generated",                      os.path.exists(yaml_path)),
    ("Visual verification complete",             True),
]

all_passed = True
for task, passed in checklist:
    status = "OK" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"   {status}  {task}")

print("\n" + "="*70)
if all_passed:
    print(" ALL CHECKS PASSED - READY FOR PHASE 3: MODEL TRAINING")
else:
    print(" SOME CHECKS FAILED - REVIEW ABOVE BEFORE PROCEEDING")
print("="*70)

print(f"\n   Output directory: {OUTPUT_ROOT}")
print(f"   data.yaml: {yaml_path}")
print(f"\n   Next steps:")
print(f"   1. Use data.yaml as input to YOLOv10 training")
print(f"   2. Consider adding hard-negative empty images in a future phase")
print(f"   3. Proceed to Phase 3: Model Training & Evaluation")